In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [24]:
file_path = "../data/raw/Component_One_Core_Feasibility.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1200, 25)


,case_id,business_stage,business_category,business_description,district,province,location_type,proposed_action,available_capital_lkr,loan_amount_lkr,...,customer_demand_score,expected_operating_days_per_month,feasibility_label,entrepreneur_experience_years,location_suitability_score,available_staff_count,required_staff_count,available_equipment_score,required_equipment_score,supplier_availability_score
0,C00001,New,Grocery/Mini-Mart,Rise selling,Colombo,Western,Diyagama,start,730108,0,...,56,25,Conditionally Feasible,1,2,1,0,1,1,4
1,C00002,New,Grocery/Mini-Mart,Spices,Colombo,Western,Siyambalagoda,start,97238,28234,...,54,26,Infeasible,1,2,0,2,1,2,4
2,C00003,New,Grocery/Mini-Mart,Baking Bread & sweets,Colombo,Western,Habarakada,start,213530,0,...,91,27,Conditionally Feasible,5,4,1,0,1,1,4
3,C00004,New,Grocery/Mini-Mart,Tea selling,Colombo,Western,Jalthara,start,208402,20000,...,67,27,Conditionally Feasible,0,4,1,0,2,2,4
4,C00005,New,Grocery/Mini-Mart,Cool Drinks,Colombo,Western,Siyambalagoda,start,187575,37931,...,83,27,Conditionally Feasible,0,4,0,1,1,2,4


In [25]:
# Target variable
y = df["feasibility_label"]

# Remove target, ID, and free-text description
X = df.drop(columns=[
    "feasibility_label",
    "case_id",
    "business_description"
])

print("Input features:", X.shape)
print("Target:", y.shape)

Input features: (1200, 22)
Target: (1200,)


In [26]:
categorical_features = [
    "business_stage",
    "business_category",
    "district",
    "province",
    "location_type",
    "proposed_action",
    "competition_level"
]

numerical_features = [
    "available_capital_lkr",
    "loan_amount_lkr",
    "monthly_budget_lkr",
    "initial_inventory_cost_lkr",
    "expected_price_lkr",
    "expected_customers_per_day",
    "customer_demand_score",
    "expected_operating_days_per_month",
    "entrepreneur_experience_years",
    "location_suitability_score",
    "available_staff_count",
    "required_staff_count",
    "available_equipment_score",
    "required_equipment_score",
    "supplier_availability_score"
]

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

Categorical features: 7
Numerical features: 15


In [27]:
categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [28]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (840, 22)
Validation: (180, 22)
Testing: (180, 22)


In [29]:
# Fit preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and test data using the fitted preprocessor
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (840, 76)
Processed validation shape: (180, 76)
Processed testing shape: (180, 76)


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Automatically identify numeric columns
numeric_features = X_train.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

# Automatically identify categorical/text columns
categorical_features = X_train.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# Numeric preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessing to validation and test data
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


print("\nPreprocessing completed successfully!")
print("X_train shape:", X_train_processed.shape)
print("X_val shape:", X_val_processed.shape)
print("X_test shape:", X_test_processed.shape)

Numeric features:
['available_capital_lkr', 'loan_amount_lkr', 'monthly_budget_lkr', 'initial_inventory_cost_lkr', 'expected_price_lkr', 'expected_customers_per_day', 'customer_demand_score', 'expected_operating_days_per_month', 'entrepreneur_experience_years', 'location_suitability_score', 'available_staff_count', 'required_staff_count', 'available_equipment_score', 'required_equipment_score', 'supplier_availability_score']

Categorical features:
['business_stage', 'business_category', 'district', 'province', 'location_type', 'proposed_action', 'competition_level']

Preprocessing completed successfully!
X_train shape: (840, 76)
X_val shape: (180, 76)
X_test shape: (180, 76)


C:\Users\DELL\AppData\Local\Temp\ipykernel_7144\2342246397.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [31]:
# Fit preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and test data using the fitted preprocessor
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (840, 76)
Processed validation shape: (180, 76)
Processed testing shape: (180, 76)


In [32]:
import numpy as np
import pandas as pd

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

# Check NaN values
print("NaN values in training:", np.isnan(to_dense(X_train_processed)).sum())
print("NaN values in validation:", np.isnan(to_dense(X_val_processed)).sum())
print("NaN values in testing:", np.isnan(to_dense(X_test_processed)).sum())


NaN values in training: 0
NaN values in validation: 0
NaN values in testing: 0


In [33]:
import numpy as np

def count_nan(X):
    if hasattr(X, "toarray"):
        X = X.toarray()
    return np.isnan(X).sum()

print("NaN values in training:", count_nan(X_train_processed))
print("NaN values in validation:", count_nan(X_val_processed))
print("NaN values in testing:", count_nan(X_test_processed))

NaN values in training: 0
NaN values in validation: 0
NaN values in testing: 0


In [34]:
def check_data(X, name):
    if hasattr(X, "toarray"):
        X = X.toarray()

    print(f"\n{name}")
    print("Shape:", X.shape)
    print("NaN values:", np.isnan(X).sum())
    print("Infinite values:", np.isinf(X).sum())

check_data(X_train_processed, "Training")
check_data(X_val_processed, "Validation")
check_data(X_test_processed, "Testing")


Training
Shape: (840, 76)
NaN values: 0
Infinite values: 0

Validation
Shape: (180, 76)
NaN values: 0
Infinite values: 0

Testing
Shape: (180, 76)
NaN values: 0
Infinite values: 0


In [36]:
import numpy as np
import pandas as pd

# Safe conversion helper: handles both sparse matrices and dense ndarrays
def to_dense_array(X):
    fn = getattr(X, "toarray", None)
    return fn() if callable(fn) else np.asarray(X)

X_train_processed_array = to_dense_array(X_train_processed)
X_val_processed_array = to_dense_array(X_val_processed)
X_test_processed_array = to_dense_array(X_test_processed)

# Create DataFrames with the correct feature names
X_train_processed_df = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train.index
)

X_val_processed_df = pd.DataFrame(
    X_val_processed_array,
    columns=feature_names,
    index=X_val.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed_array,
    columns=feature_names,
    index=X_test.index
)

print("Training:", X_train_processed_df.shape)
print("Validation:", X_val_processed_df.shape)
print("Testing:", X_test_processed_df.shape)


Training: (840, 76)
Validation: (180, 76)
Testing: (180, 76)


In [37]:
print(X_train_processed[:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 110 stored elements and shape (5, 76)>
  Coords	Values
  (0, 0)	-1.4520233872200818
  (0, 1)	0.7326439413121238
  (0, 2)	-1.6369986133407521
  (0, 3)	-0.9838674171489787
  (0, 4)	-0.7565432031637862
  (0, 5)	-1.3259376366853248
  (0, 6)	0.5018942542536136
  (0, 7)	0.3701663732545886
  (0, 8)	0.46177607283599986
  (0, 9)	-0.5785590479359872
  (0, 10)	-0.8632054383495057
  (0, 11)	-1.0727338374975637
  (0, 12)	-1.0314602194023625
  (0, 13)	-1.6043995880704829
  (0, 14)	-1.4849090209055247
  (0, 16)	1.0
  (0, 19)	1.0
  (0, 21)	1.0
  (0, 22)	1.0
  (0, 43)	1.0
  (0, 72)	1.0
  (0, 75)	1.0
  (1, 0)	-1.0274248130699943
  (1, 1)	-0.47894693124968163
  (1, 2)	-0.11588239568677232
  :	:
  (3, 27)	1.0
  (3, 69)	1.0
  (3, 75)	1.0
  (4, 0)	0.6497805360120514
  (4, 1)	-0.47894693124968163
  (4, 2)	0.07975956156775885
  (4, 3)	0.03399909045587639
  (4, 4)	-0.12548953036272514
  (4, 5)	-0.7152080129447574
  (4, 6)	0.5832825117001454
  (4, 7)

In [15]:
print("Original features:", X_train.shape[1])
print("Processed features:", X_train_processed.shape[1])

Original features: 22
Processed features: 76


In [16]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 76
['num__available_capital_lkr' 'num__loan_amount_lkr'
 'num__monthly_budget_lkr' 'num__initial_inventory_cost_lkr'
 'num__expected_price_lkr' 'num__expected_customers_per_day'
 'num__customer_demand_score' 'num__expected_operating_days_per_month'
 'num__entrepreneur_experience_years' 'num__location_suitability_score'
 'num__available_staff_count' 'num__required_staff_count'
 'num__available_equipment_score' 'num__required_equipment_score'
 'num__supplier_availability_score' 'cat__business_stage_Existing'
 'cat__business_stage_New' 'cat__business_category_Bakery/Food/Grocery'
 'cat__business_category_Beauty Saloon'
 'cat__business_category_Clothing/ Garment'
 'cat__business_category_Grocery/Mini-Mart' 'cat__district_Colombo'
 'cat__province_Western' 'cat__location_type_Ambalangoda'
 'cat__location_type_Arukwatta' 'cat__location_type_Bope'
 'cat__location_type_Brahamnagama' 'cat__location_type_Diyagama'
 'cat__location_type_Dolahena' 'cat__location_type_Go

In [17]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_val_processed shape:", X_val_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

print("Number of feature names:", len(feature_names))

print("Type:", type(X_train_processed))

X_train_processed shape: (840, 76)
X_val_processed shape: (180, 76)
X_test_processed shape: (180, 76)
Number of feature names: 76
Type: <class 'scipy.sparse._csr.csr_matrix'>


In [38]:
# Instead of: print(X_train_processed[:5])
print(X_train_processed[:5].toarray())

# Instead of: print(X_train_processed[0])
print(X_train_processed[0].toarray())


[[-1.45202339  0.73264394 -1.63699861 -0.98386742 -0.7565432  -1.32593764
   0.50189425  0.37016637  0.46177607 -0.57855905 -0.86320544 -1.07273384
  -1.03146022 -1.60439959 -1.48490902  0.          1.          0.
   0.          1.          0.          1.          1.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          1.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   1.          0.          0.          1.        ]
 [-1.02742481 -0.47894693 -0.1158824  -0.79323081 -0.69596205 -0.47091616
  -0.96309438  1.25351795 -0.86966066 -0.57855905 -0.86320544 -0.01

In [39]:
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['available_capital_lkr', 'loan_amount_lkr',
                                  'monthly_budget_lkr',
                                  'initial_inventory_cost_lkr',
                                  'expected_price_lkr',
                                  'expected_customers_per_day',
                                  'customer_demand_score',
                                  'expected_operating_days_per_month',
                                  'entrepreneur_exp...
                                  'required_staff_count',
                                  'available_equipment_score',
                                  'required_equipment_score',
                                  'supplier_ava

In [40]:
# Convert sparse matrices to dense arrays
X_train_processed_array = X_train_processed.toarray()
X_val_processed_array = X_val_processed.toarray()
X_test_processed_array = X_test_processed.toarray()

# Create DataFrames with the correct feature names
X_train_processed_df = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train.index
)

X_val_processed_df = pd.DataFrame(
    X_val_processed_array,
    columns=feature_names,
    index=X_val.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed_array,
    columns=feature_names,
    index=X_test.index
)

print("Training:", X_train_processed_df.shape)
print("Validation:", X_val_processed_df.shape)
print("Testing:", X_test_processed_df.shape)

Training: (840, 76)
Validation: (180, 76)
Testing: (180, 76)


In [21]:
print("Training NaN:", X_train_processed_df.isna().sum().sum())
print("Validation NaN:", X_val_processed_df.isna().sum().sum())
print("Testing NaN:", X_test_processed_df.isna().sum().sum())

Training NaN: 0
Validation NaN: 0
Testing NaN: 0
